# Notebook 03 — Claims Intelligence Analytics

This notebook demonstrates:
1. Synthetic claims data generation
2. Claims classification and resolution analysis
3. Root cause analysis (damage by zone / SKU / carrier / shift)
4. Inconsistency detection performance
5. Financial impact summary (cost recovered vs. avoided)
6. Dashboard data payload preview

In [ ]:
import sys
from pathlib import Path

PROJECT_ROOT = Path('..').resolve()
sys.path.insert(0, str(PROJECT_ROOT))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
from datetime import datetime, timedelta

plt.rcParams['figure.dpi'] = 120
plt.rcParams['axes.spines.top'] = False
plt.rcParams['axes.spines.right'] = False

## 1. Generate Synthetic Claims Data

In [ ]:
rng = np.random.default_rng(42)

N = 500
carriers = ['FXFE', 'ODFL', 'SAIA', 'RDWY', 'CEVA']
skus = ['PHARMA', 'ELECTRONICS', 'GLASS', 'FROZEN_FOOD', 'DRY_GOODS']
zones = ['pharma_vault', 'frozen_storage', 'loading_dock', 'ambient_storage']
shifts = ['morning', 'afternoon', 'night']
severities = ['none', 'minor', 'moderate', 'severe']
origins = ['pre_existing', 'warehouse', 'carrier', 'unknown']

base_date = datetime(2024, 1, 1)
dates = [base_date + timedelta(days=rng.integers(0, 365)) for _ in range(N)]

claims = pd.DataFrame({
    'claim_id': [f'CLM-{i:05d}' for i in range(N)],
    'date': sorted(dates),
    'carrier': rng.choice(carriers, N, p=[0.30, 0.25, 0.20, 0.15, 0.10]),
    'sku_category': rng.choice(skus, N, p=[0.20, 0.25, 0.10, 0.25, 0.20]),
    'zone': rng.choice(zones, N),
    'shift': rng.choice(shifts, N, p=[0.40, 0.35, 0.25]),
    'damage_severity': rng.choice(severities, N, p=[0.05, 0.40, 0.35, 0.20]),
    'damage_origin': rng.choice(origins, N, p=[0.20, 0.35, 0.30, 0.15]),
    'claim_amount': rng.lognormal(mean=5.5, sigma=1.2, size=N).round(2),
    'auto_resolved': rng.choice([True, False], N, p=[0.67, 0.33]),
    'fraud_flagged': rng.choice([True, False], N, p=[0.08, 0.92]),
})

print(f'Generated {len(claims):,} synthetic claims')
claims.head()

## 2. Claims Resolution Overview

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 5))

# Damage origin distribution
origin_counts = claims['damage_origin'].value_counts()
axes[0].pie(origin_counts.values, labels=origin_counts.index, autopct='%1.1f%%', startangle=90)
axes[0].set_title('Damage Origin')

# Severity distribution
sev_counts = claims['damage_severity'].value_counts().reindex(severities)
axes[1].bar(sev_counts.index, sev_counts.values, color=['#2ecc71','#f1c40f','#e67e22','#e74c3c'])
axes[1].set_title('Damage Severity')
axes[1].set_ylabel('Count')

# Auto-resolution rate by severity
auto_rate = claims.groupby('damage_severity')['auto_resolved'].mean().reindex(severities) * 100
axes[2].bar(auto_rate.index, auto_rate.values, color='steelblue')
axes[2].set_title('Auto-Resolution Rate by Severity')
axes[2].set_ylabel('Auto-resolved (%)')
axes[2].axhline(60, ls='--', color='red', lw=0.8, label='60% target')
axes[2].legend()

plt.tight_layout()
plt.show()

print(f'Overall auto-resolution rate: {claims["auto_resolved"].mean()*100:.1f}%')
print(f'Fraud flagged: {claims["fraud_flagged"].sum()} ({claims["fraud_flagged"].mean()*100:.1f}%)')

## 3. Root Cause Analysis

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

def plot_damage_rate(ax, groupby_col, title):
    total = claims.groupby(groupby_col).size()
    damaged = claims[claims['damage_severity'] != 'none'].groupby(groupby_col).size()
    rate = (damaged / total * 100).sort_values(ascending=False)
    bars = ax.bar(rate.index, rate.values, color='steelblue')
    ax.axhline(5.0, ls='--', color='red', lw=0.8, label='5% alert threshold')
    # Highlight bars above threshold
    for bar, val in zip(bars, rate.values):
        if val >= 5.0:
            bar.set_color('#e74c3c')
    ax.set_title(title)
    ax.set_ylabel('Damage Rate (%)')
    ax.tick_params(axis='x', rotation=30)
    ax.legend(fontsize=8)

plot_damage_rate(axes[0, 0], 'carrier', 'Damage Rate by Carrier')
plot_damage_rate(axes[0, 1], 'sku_category', 'Damage Rate by SKU Category')
plot_damage_rate(axes[1, 0], 'zone', 'Damage Rate by Warehouse Zone')
plot_damage_rate(axes[1, 1], 'shift', 'Damage Rate by Shift')

plt.suptitle('Root Cause Analysis — Damage Rate Breakdown', fontsize=14, y=1.01)
plt.tight_layout()
plt.show()

## 4. Financial Impact

In [ ]:
total_claims = claims['claim_amount'].sum()
auto_resolved = claims[claims['auto_resolved']]['claim_amount'].sum()
fraud_prevented = claims[claims['fraud_flagged']]['claim_amount'].sum()
warehouse_liability = claims[claims['damage_origin'] == 'warehouse']['claim_amount'].sum()
carrier_liability = claims[claims['damage_origin'] == 'carrier']['claim_amount'].sum()
rejected = claims[claims['damage_origin'].isin(['pre_existing'])]['claim_amount'].sum()

summary = {
    'Total Claim Value': total_claims,
    'Auto-Resolved': auto_resolved,
    'Fraud Prevented': fraud_prevented,
    'Warehouse Liability': warehouse_liability,
    'Carrier Recovery': carrier_liability,
    'Rejected (Pre-existing)': rejected,
}

print('=== Financial Impact Summary ===')
for k, v in summary.items():
    print(f'  {k:<30} ${v:>12,.2f}')

fig, ax = plt.subplots(figsize=(10, 5))
keys = list(summary.keys())
vals = list(summary.values())
colors = ['#3498db', '#2ecc71', '#e74c3c', '#e67e22', '#9b59b6', '#95a5a6']
ax.barh(keys, vals, color=colors)
ax.set_xlabel('USD')
ax.xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'${x:,.0f}'))
ax.set_title('Financial Impact Breakdown')
plt.tight_layout()
plt.show()

## 5. 30-Day Damage Trend

In [ ]:
claims['month'] = pd.to_datetime(claims['date']).dt.to_period('M')
monthly = claims.groupby('month').agg(
    total_claims=('claim_id', 'count'),
    total_amount=('claim_amount', 'sum'),
    auto_resolved_count=('auto_resolved', 'sum'),
    fraud_flagged_count=('fraud_flagged', 'sum'),
).reset_index()
monthly['auto_resolution_rate'] = monthly['auto_resolved_count'] / monthly['total_claims'] * 100

fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(14, 8), sharex=True)

ax1.bar(monthly['month'].astype(str), monthly['total_claims'], color='steelblue')
ax1.set_title('Monthly Claims Volume')
ax1.set_ylabel('Claims Count')

ax2.plot(monthly['month'].astype(str), monthly['auto_resolution_rate'],
         color='green', marker='o', lw=2, label='Auto-resolution rate')
ax2.axhline(67, ls='--', color='green', lw=0.8, alpha=0.5, label='Target 67%')
ax2.set_ylabel('Auto-Resolution Rate (%)')
ax2.set_xlabel('Month')
ax2.tick_params(axis='x', rotation=45)
ax2.legend()

plt.tight_layout()
plt.show()

## 6. Live Pipeline Test

In [ ]:
from src.claims.classifier import ClaimClassifier, ClaimValidity, LiabilityParty
from src.claims.inconsistency import InconsistencyDetector
from src.evidence.comparator import ConditionComparator, DamageOrigin

comp = ConditionComparator(use_clip=False)
clf = ClaimClassifier()
det = InconsistencyDetector()

test_cases = [
    {'sscc': 'SSCC001', 'inbound': 'none', 'outbound': 'moderate', 'carrier': False, 'amount': 450.0},
    {'sscc': 'SSCC002', 'inbound': 'minor', 'outbound': 'minor',   'carrier': True,  'amount': 85.0},
    {'sscc': 'SSCC003', 'inbound': 'none', 'outbound': 'severe',   'carrier': False, 'amount': 12000.0},
]

print(f'{"SSCC":<12} {"Origin":<20} {"Validity":<12} {"Priority":<10} {"Action"}')
print('-' * 72)
for tc in test_cases:
    comp_result = comp.compare(tc['sscc'], tc['inbound'], tc['outbound'], carrier_reported=tc['carrier'])
    clf_result = clf.classify(
        claim_id=f'CLM-{tc["sscc"]}',
        damage_origin=comp_result.damage_origin,
        inbound_severity=tc['inbound'],
        outbound_severity=tc['outbound'],
        claim_amount=tc['amount'],
    )
    action = 'AUTO-APPROVE' if clf_result.auto_approve else ('ESCALATE' if clf_result.escalate_to_human else 'REVIEW')
    print(f'{tc["sscc"]:<12} {comp_result.damage_origin:<20} {clf_result.validity:<12} {clf_result.priority:<10} {action}')